# ContractRisk — corrección 02 de integración

Sesión nueva de Colab con GPU T4. Ejecutar en orden. No modifica el Space actual ni usa el gold final.

Prueba transcripción visual + extracción determinista de campos, control de imágenes uniformes y elegibilidad de la herramienta. Reutiliza las cuatro imágenes originales con hashes verificados, no repite M1 y carga el generador una sola vez después de liberar el VLM.

La orquestación visual reutiliza las lecturas guardadas: no es latencia fresca punta a punta ni benchmark de eficiencia. Se distinguen errores, formato válido, control de píxeles y llamadas reales del VLM. Calidad sujeta a revisión humana.

Para W&B: habilitar el secreto WANDB_API_KEY. Sin él se conservan resultados locales. La última celda descarga evidencia incluso si falló la ejecución.

In [ ]:
import os, sys, subprocess, tempfile, json
from pathlib import Path

ROOT = Path(tempfile.mkdtemp(prefix='contractrisk_final_correction02_', dir='/content')) / 'repo'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'Entrega/contractrisk-final',
    'https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git', str(ROOT)], check=True)
os.chdir(ROOT)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', COMMIT)

def run_visible(arguments, logfile):
    with open(logfile, 'w', encoding='utf-8') as log:
        p = subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        status = p.wait()
    if status:
        raise RuntimeError(f'Proceso terminó con código {status}; consultar {logfile}')

try:
    from google.colab import userdata
    key = userdata.get('WANDB_API_KEY')
    if key:
        os.environ['WANDB_API_KEY'] = key
    del key
    print('Secreto W&B disponible; no se imprime su valor.')
except Exception:
    print('Sin secreto W&B accesible: la prueba guardará evidencia local.')


In [ ]:
run_visible([sys.executable, '-m', 'pip', 'install', '-r',
    'final/requirements-integration.txt'], 'installation.log')
import torch
run_visible([sys.executable, '-c', 'from final.run_integration import fixture_font_path; print(fixture_font_path().name)'], 'font_check.log')
assert torch.cuda.is_available(), 'Selecciona GPU T4 y una sesión nueva.'
print('GPU:', torch.cuda.get_device_name(0))
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'final/tests', '-v'], 'tests_final.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], 'tests_m3.log')


In [ ]:
RESULTS = ROOT / 'correction02_development_results'
run_visible([sys.executable, '-u', 'final/run_correction02.py', '--output', str(RESULTS)], 'correction02.log')


## Inspección y conservación de resultados

Ejecutar las celdas siguientes también si falla la integración. `quality_approved=false` es deliberado: las lecturas, fuentes y respuestas necesitan revisión humana.

Los eventos de W&B de esta prueba deben quedar separados de la futura evaluación final y finalizar con `finish()`. Si la conexión falla, conservar la evidencia local; no publicar la clave.


In [ ]:
RESULTS = ROOT / 'correction02_development_results'
metadata = RESULTS / 'metadata.json'
print(metadata.read_text(encoding='utf-8') if metadata.exists() else 'Sin metadata: revisar installation.log/correction02.log.')
rows_path = RESULTS / 'correction_rows.jsonl'
if rows_path.exists():
    for line in rows_path.read_text(encoding='utf-8').splitlines():
        row = json.loads(line)
        result = row.get('result', {})
        print(row['stage'], row['case_id'], row.get('error', ''),
            json.dumps(result.get('output', result.get('visual', {})), ensure_ascii=False))


In [ ]:
import shutil
from google.colab import files
RESULTS.mkdir(exist_ok=True)
(RESULTS / 'notebook_commit.txt').write_text(COMMIT + '\n', encoding='utf-8')
for log in ROOT.glob('*.log'):
    shutil.copy2(log, RESULTS / log.name)
package = shutil.make_archive(str(ROOT / 'contractrisk_final_correccion_02_resultados'), 'zip', RESULTS)
files.download(package)
